In [3]:
# setup & config ---------------------------------------------------------
# (Lightkurve pulls Kepler data from MAST; requires internet at runtime.)

# !pip -q install lightkurve astroquery astropy pywavelets

import os
import math
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

try:
    import lightkurve as lk
    from astropy.timeseries import BoxLeastSquares
except Exception as e:
    print("Note: lightkurve/astropy not imported yet. Install them if you plan to fetch from MAST.")
    print(e)

from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

try:
    import pywt
except Exception as e:
    print("Scalogram (pywt) optional. Install pywavelets if you want it.")
    print(e)

data_path = "data/Kepler Object of Interest.xlsx"  # << change to your file
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True, parents=True)

TOP_N = 5


c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [12]:
# fetch & preprocess light curves for koi

def fetch_pdcsap_lightcurve(kic_id: int, mission: str):
    """Download and return a stitched PDCSAP light curve for a given Kepler KIC ID."""
    try:
        sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission=mission)
        if len(sr) == 0:
            print(f"No LC files found for KIC {kic_id}.")
            return None
        lcf = sr.download_all()
        if lcf is None:
            print(f"Download failed for KIC {kic_id}.")
            return None
        lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
        lc = lc.flatten(window_length=401)
        return lc
    except Exception as e:
        print(f"Error fetching LC for KIC {kic_id}: {e}")
        return None


In [ ]:
# libraries
import pandas as pd

# create a json structure with two id fields 
kepid = [
  {"Kepid": 3118797, "KepoiName": "K01950.01"},
  {"Kepid": 4254466, "KepoiName": "K02134.01"},
  {"Kepid": 4914423, "KepoiName": "K00108.02"},
  {"Kepid": 6041734, "KepoiName": "K02167.01"},
  {"Kepid": 6047072, "KepoiName": "K02103.01"},
  {"Kepid": 6541920, "KepoiName": "K00157.06"},
  {"Kepid": 6665512, "KepoiName": "K02005.01"},
  {"Kepid": 7045496, "KepoiName": "K01939.01"},
  {"Kepid": 7094486, "KepoiName": "K01907.01"},
  {"Kepid": 7449541, "KepoiName": "K02063.01"},
  {"Kepid": 7620413, "KepoiName": "K02103.01"},
  {"Kepid": 7918992, "KepoiName": "K02095.01"},
  {"Kepid": 8081187, "KepoiName": "K01951.01"},
  {"Kepid": 8164012, "KepoiName": "K02116.01"},
  {"Kepid": 8233702, "KepoiName": "K02140.01"},
  {"Kepid": 8395660, "KepoiName": "K00116.02"},
  {"Kepid": 8559644, "KepoiName": "K00139.02"},
  {"Kepid": 8559644, "KepoiName": "K00139.01"},
  {"Kepid": 9006186, "KepoiName": "K02169.03"},
  {"Kepid": 9072190, "KepoiName": "K01933.01"},
  {"Kepid": 9205938, "KepoiName": "K02162.02"},
  {"Kepid": 9349482, "KepoiName": "K02020.01"},
  {"Kepid": 9353314, "KepoiName": "K01900.01"},
  {"Kepid": 9471974, "KepoiName": "K00119.01"},
  {"Kepid": 9473078, "KepoiName": "K02079.01"},
  {"Kepid": 9579641, "KepoiName": "K00115.01"},
  {"Kepid": 9790806, "KepoiName": "K02035.01"},
  {"Kepid": 10187017, "KepoiName": "K00082.04"},
  {"Kepid": 10190777, "KepoiName": "K01937.01"},
  {"Kepid": 10857519, "KepoiName": "K02075.01"},
  {"Kepid": 10875245, "KepoiName": "K00117.04"},
  {"Kepid": 10965588, "KepoiName": "K02177.01"},
  {"Kepid": 11086270, "KepoiName": "K00124.02"},
  {"Kepid": 11614617, "KepoiName": "K01990.01"},
  {"Kepid": 11923284, "KepoiName": "K02026.01"},
  {"Kepid": 12154526, "KepoiName": "K02004.01"}
]

# turn into a dataframe
kepid_df = pd.DataFrame(kepid)

# bring in the koi data
# slice the columns to kepid, kepoiname, koi_period, and koi_time0bk
koi_data_path = "data/Kepler Object of Interest.csv" 
koi_df = pd.read_csv(koi_data_path, usecols=["kepid", "kepoi_name", "koi_period", "koi_time0bk"])

# merge the two dataframes on kepoiname
enriched_koi_df = pd.merge(kepid_df, koi_df, left_on="KepoiName", right_on="kepoi_name", how="left")

# slice the columns to kepid, kepoiname, koi_period, and koi_time0bk
enriched_koi_df = enriched_koi_df[["kepid", "KepoiName", "koi_period", "koi_time0bk"]]
enriched_koi_df.head()

,kepid,KepoiName,koi_period,koi_time0bk
0,3118797,K01950.01,66.415979,188.139880
1,4254466,K02134.01,42.300628,168.777830
2,4914423,K00108.02,179.609803,295.330361
3,6041734,K02167.01,24.339820,136.058130
4,7620413,K02103.01,2.543064,132.681590


In [19]:
# tid 
tid = [
  {"TID": 351601843, "TOI": 1075.01},
  {"TID": 394561119, "TOI": 1107.01},
  {"TID": 52368076, "TOI": 125.02},
  {"TID": 355867695, "TOI": 1260.02},
  {"TID": 417948359, "TOI": 1272.01},
  {"TID": 89020549, "TOI": 132.01},
  {"TID": 62483237, "TOI": 139.01},
  {"TID": 428679607, "TOI": 1669.01},
  {"TID": 28900646, "TOI": 1685.01},
  {"TID": 183120439, "TOI": 169.01},
  {"TID": 233602827, "TOI": 1749.02},
  {"TID": 376524552, "TOI": 1811.01},
  {"TID": 404505029, "TOI": 1842.01},
  {"TID": 27491137, "TOI": 2076.01},
  {"TID": 441738827, "TOI": 2084.01},
  {"TID": 235678745, "TOI": 2095.02},
  {"TID": 392476080, "TOI": 2109.01},
  {"TID": 88992642, "TOI": 2145.01},
  {"TID": 395393265, "TOI": 2152.01},
  {"TID": 198485881, "TOI": 2257.01},
  {"TID": 24358417, "TOI": 2338.01},
  {"TID": 149845414, "TOI": 2545.01},
  {"TID": 348835438, "TOI": 2669.01},
  {"TID": 178162579, "TOI": 2842.01},
  {"TID": 361343239, "TOI": 2977.01},
  {"TID": 458419328, "TOI": 3785.01},
  {"TID": 95660472, "TOI": 3819.01},
  {"TID": 95057860, "TOI": 4201.01},
  {"TID": 144193715, "TOI": 4308.01},
  {"TID": 126606859, "TOI": 4479.01},
  {"TID": 49428710, "TOI": 5174.01},
  {"TID": 148673433, "TOI": 5704.01}
]

# turn into a dataframe
toi_df = pd.DataFrame(tid)

# bring in the tess data
tess_data_path = "data/TESS Project Candidates.csv"
tess_df = pd.read_csv(tess_data_path, usecols=["tid", "toi", "pl_orbper", "pl_tranmid"])

# merge the two dataframes on toi
enriched_tess_df = pd.merge(toi_df, tess_df, left_on="TOI", right_on="toi", how="left")

# slice the columns to kepid, kepoiname, koi_period, and koi_time0bk
enriched_tess_df = enriched_tess_df[["tid", "toi", "pl_orbper", "pl_tranmid"]]
enriched_tess_df.head()


,tid,toi,pl_orbper,pl_tranmid
0,351601843,1075.01,0.604739,2458654.250
1,394561119,1107.01,4.078239,2459336.074
2,52368076,125.02,9.154942,2460202.050
3,355867695,1260.02,7.493161,2458686.120
4,417948359,1272.01,3.315977,2459661.401


In [20]:
# view enriched df info
enriched_tess_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   tid         32 non-null     int64  
 1   toi         32 non-null     float64
 2   pl_orbper   32 non-null     float64
 3   pl_tranmid  32 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 1.1 KB


In [7]:
# --- deps ---
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation, PillowWriter
import lightkurve as lk

# optional: where to save outputs
out_dir = Path("gifs/koi"); out_dir.mkdir(parents=True, exist_ok=True)

def fetch_pdcsap_lightcurve(kic_id: int):
    """Download and return a stitched, flattened PDCSAP light curve for a Kepler KIC ID."""
    try:
        sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
        if len(sr) == 0:
            print(f"No LC files found for KIC {kic_id}.")
            return None
        lcf = sr.download_all()
        if lcf is None:
            print(f"Download failed for KIC {kic_id}.")
            return None
        lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
        lc = lc.flatten(window_length=401)
        return lc
    except Exception as e:
        print(f"Error fetching LC for KIC {kic_id}: {e}")
        return None

def make_orbit_light_animation_3d(
    kic: int, lc, period: float, t0: float,
    frames: int = 300, orbit_radius: float = 3.0, save_path=None,
    rotate_camera=True, rot_per_anim=0.6, elev_deg=22,
    close_after=False
):
    """
    3D orbit (top) + measured PDCSAP flux vs phase (bottom), synchronized.
    """
    if lc is None:
        print("No light curve available for animation.")
        return None

    # --- Prepare arrays (plain floats, handle masked/quantities) ---
    t = np.asarray(getattr(lc.time, "value", lc.time), dtype=float)
    y = getattr(lc, "flux", None)
    if y is None:
        print("No flux in light curve.")
        return None
    y = getattr(y, "value", y)
    y = np.ma.getdata(y).astype(float, copy=False)

    # drop non-finite pairs up-front
    m = np.isfinite(t) & np.isfinite(y)
    t, y = t[m], y[m]

    if not np.isfinite(period) or period <= 0 or t.size < 10 or y.size < 10:
        print("Invalid LC/period for animation.")
        return None

    # normalize flux to ~1.0 median (helps alpha mapping)
    med = np.nanmedian(y)
    if np.isfinite(med) and med != 0:
        y = y / med

    # Phase-fold & sort for smooth traversal (BKJD compatible)
    t_rel = (t - float(t0)) % float(period)
    order = np.argsort(t_rel)
    t_rel, y = t_rel[order], y[order]
    if t_rel.size < 10:
        print("Too few samples to animate.")
        return None

    # Robust brightness -> alpha mapping
    finite_y = y[np.isfinite(y)]
    if finite_y.size < 10:
        print("Too few finite flux samples to animate.")
        return None
    q1, q2 = np.nanpercentile(finite_y, [1, 99])
    span = q2 - q1 if np.isfinite(q2 - q1) else 0.0

    def flux_to_alpha(f):
        if not np.isfinite(f) or span < 1e-8:
            return 1.0
        x = (float(f) - q1) / span
        return float(np.clip(0.3 + 0.7 * x, 0.0, 1.0))

    frames_eff = int(min(frames, max(20, t_rel.size)))

    # --- Figure layout (3D top + 2D bottom) ---
    fig = plt.figure(figsize=(7.2, 8.4))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0])

    ax3d = fig.add_subplot(gs[0], projection='3d')
    axlc = fig.add_subplot(gs[1])

    # 3D orbit scaffolding
    th = np.linspace(0, 2*np.pi, 600)
    x_orb = orbit_radius*np.cos(th)
    y_orb = orbit_radius*np.sin(th)
    z_orb = np.zeros_like(th)
    ax3d.plot(x_orb, y_orb, z_orb, lw=1.5)

    # Star (scatter so we can set alpha dynamically)
    star = ax3d.scatter([0],[0],[0], s=600, alpha=1.0)

    # Planet marker (updated every frame)
    planet = ax3d.scatter([orbit_radius],[0],[0], s=60)

    # Nice view box
    lim = orbit_radius*1.25
    ax3d.set_xlim(-lim, lim); ax3d.set_ylim(-lim, lim); ax3d.set_zlim(-lim*0.2, lim*0.2)
    ax3d.set_xlabel("x"); ax3d.set_ylabel("y"); ax3d.set_zlabel("z")
    ax3d.set_title(f"KIC {kic} — schematic 3D orbit", pad=12)
    ax3d.view_init(elev=elev_deg, azim=45)

    # Flux plot
    axlc.plot(t_rel, y, lw=0.6)
    tracker = axlc.axvline(0.0, ls="--", lw=1.0)
    axlc.set_xlim(0, float(period))
    axlc.set_xlabel("Phase within period [days]")
    axlc.set_ylabel("Flux (normalized)")
    axlc.set_title("Measured flux vs phase (PDCSAP)")

    def update(i):
        # Index into folded series
        idx = int((i / max(1, (frames_eff - 1))) * (t_rel.size - 1))
        phase_now = float(t_rel[idx])
        angle = (phase_now / float(period)) * 2*np.pi

        # Planet position on circular orbit (in x–y plane)
        xp = orbit_radius*np.cos(angle)
        yp = orbit_radius*np.sin(angle)
        zp = 0.0

        # Update artists
        planet._offsets3d = ([xp], [yp], [zp])  # updating Path3DCollection
        tracker.set_xdata([phase_now, phase_now])

        # Star brightness follows flux
        a = flux_to_alpha(y[idx])
        star.set_alpha(a)

        # Optional camera rotation for depth perception
        if rotate_camera:
            az = 45 + 360*rot_per_anim*(i/frames_eff)
            ax3d.view_init(elev=elev_deg, azim=az)

        return planet, tracker, star

    anim = FuncAnimation(fig, update, frames=frames_eff, interval=35, blit=False)
    plt.tight_layout()

    if save_path:
        try:
            anim.save(save_path, writer=PillowWriter(fps=25))
            print("Saved animation:", save_path)
        except Exception as e:
            print("Could not save 3D animation. Error:", e)

    if close_after:
        plt.close(fig)

    return anim




In [8]:
# --- Example run for all KOIs in enriched_df ---
# enriched_df must contain: kepid, koi_period, koi_time0bk
if len(enriched_df) > 0:
    for i, row in enriched_df.iterrows():
        kic = int(row["kepid"])
        p = float(row["koi_period"])
        t0 = float(row["koi_time0bk"])

        lc = fetch_pdcsap_lightcurve(kic)
        print(f"Building 3D animation for KIC {kic} ({i+1}/{len(enriched_df)}) ...")
        gif3d_path = str((out_dir / f"{kic}.gif").resolve())

        anim3d = make_orbit_light_animation_3d(
            kic, lc, p, t0,
            frames=300,
            save_path=gif3d_path,
            close_after=True  # closes figure after saving to speed up loop
        )

        print(f"Saved GIF for KIC {kic} → {gif3d_path}")


C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:16: LightkurveDeprecationWarning: The search_lightcurvefile function is deprecated and may be removed in a future version.
        Use search_lightcurve() instead.
  sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:24: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\collections.py:163: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  return LightCurveCollection([lc.PDCSAP_FLUX for lc in self])


Building 3D animation for KIC 3118797 (1/36) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\3118797.gif
Saved GIF for KIC 3118797 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\3118797.gif


C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:16: LightkurveDeprecationWarning: The search_lightcurvefile function is deprecated and may be removed in a future version.
        Use search_lightcurve() instead.
  sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:24: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\collections.py:163: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  return LightCurveCollection([lc.PDCSAP_FLUX for lc in self])


Building 3D animation for KIC 4254466 (2/36) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\4254466.gif
Saved GIF for KIC 4254466 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\4254466.gif


C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:16: LightkurveDeprecationWarning: The search_lightcurvefile function is deprecated and may be removed in a future version.
        Use search_lightcurve() instead.
  sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:24: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\collections.py:163: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  return LightCurveCollection([lc.PDCSAP_FLUX for lc in self])


...

Building 3D animation for KIC 11614617 (34/36) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11614617.gif
Saved GIF for KIC 11614617 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11614617.gif


C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:16: LightkurveDeprecationWarning: The search_lightcurvefile function is deprecated and may be removed in a future version.
        Use search_lightcurve() instead.
  sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:24: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\collections.py:163: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  return LightCurveCollection([lc.PDCSAP_FLUX for lc in self])


Building 3D animation for KIC 11923284 (35/36) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11923284.gif
Saved GIF for KIC 11923284 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11923284.gif


C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:16: LightkurveDeprecationWarning: The search_lightcurvefile function is deprecated and may be removed in a future version.
        Use search_lightcurve() instead.
  sr = lk.search_lightcurvefile(f"KIC {kic_id}", mission="Kepler")
C:\Users\roger\AppData\Local\Temp\ipykernel_21840\3697049234.py:24: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  lc = lcf.PDCSAP_FLUX.stitch().remove_nans()
c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightkurve\collections.py:163: LightkurveDeprecationWarning: The PDCSAP_FLUX function is deprecated and may be removed in a future version.
  return LightCurveCollection([lc.PDCSAP_FLUX for lc in self])


Building 3D animation for KIC 12154526 (36/36) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12154526.gif
Saved GIF for KIC 12154526 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12154526.gif


In [26]:
# --- deps (unchanged) ---
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation, PillowWriter
import lightkurve as lk

# optional: where to save outputs
out_dir = Path("gifs/tess"); out_dir.mkdir(parents=True, exist_ok=True)

def fetch_pdcsap_lightcurve(tic_id: int):
    """Download & return a stitched, normalized, flattened TESS light curve for a TIC ID.
    Tries SPOC/QLP across common cadences; falls back to any author; then TESSCut FFIs.
    """
    try:
        # Try common authors and cadences first
        for author in ("SPOC", "QLP"):
            # None means "any cadence"; otherwise try specific exptimes too
            for exptime in (20, 120, 600, 1800, None):
                sr = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS",
                                          author=author, exptime=exptime)
                if len(sr) == 0:
                    continue
                lc_col = sr.download_all()
                if lc_col is None or len(lc_col) == 0:
                    continue
                lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
                return lc

        # Fallback: any author/cadence
        sr_any = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")
        if len(sr_any) > 0:
            lc_col = sr_any.download_all()
            if lc_col is not None and len(lc_col) > 0:
                lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
                return lc

        # Last resort: TESSCut (FFI cutouts → light curve)
        sr_cut = lk.search_tesscut(f"TIC {tic_id}")
        if len(sr_cut) > 0:
            # Download first available sector cutout and extract LC
            tpf = sr_cut[0].download(cutout_size=15)
            if tpf is not None:
                lc = (tpf.to_lightcurve(aperture_mask="pipeline")
                          .remove_nans().normalize().flatten(window_length=401))
                return lc

        print(f"No usable TESS light curve found for TIC {tic_id} (SPOC, QLP, any, TESSCut).")
        return None
    except Exception as e:
        print(f"Error fetching LC for TIC {tic_id}: {e}")
        return None


def make_orbit_light_animation_3d(
    kic: int, lc, period: float, t0: float,
    frames: int = 300, orbit_radius: float = 3.0, save_path=None,
    rotate_camera=True, rot_per_anim=0.6, elev_deg=22,
    close_after=False
):
    """
    3D orbit (top) + measured PDCSAP flux vs phase (bottom), synchronized.
    """
    if lc is None:
        print("No light curve available for animation.")
        return None

    # --- Prepare arrays (plain floats, handle masked/quantities) ---
    t = np.asarray(getattr(lc.time, "value", lc.time), dtype=float)
    y = getattr(lc, "flux", None)
    if y is None:
        print("No flux in light curve.")
        return None
    y = getattr(y, "value", y)
    y = np.ma.getdata(y).astype(float, copy=False)

    # drop non-finite pairs up-front
    m = np.isfinite(t) & np.isfinite(y)
    t, y = t[m], y[m]

    if not np.isfinite(period) or period <= 0 or t.size < 10 or y.size < 10:
        print("Invalid LC/period for animation.")
        return None

    # normalize flux to ~1.0 median (helps alpha mapping)
    med = np.nanmedian(y)
    if np.isfinite(med) and med != 0:
        y = y / med

    # Phase-fold & sort for smooth traversal (BKJD compatible)
    t_rel = (t - float(t0)) % float(period)
    order = np.argsort(t_rel)
    t_rel, y = t_rel[order], y[order]
    if t_rel.size < 10:
        print("Too few samples to animate.")
        return None

    # Robust brightness -> alpha mapping
    finite_y = y[np.isfinite(y)]
    if finite_y.size < 10:
        print("Too few finite flux samples to animate.")
        return None
    q1, q2 = np.nanpercentile(finite_y, [1, 99])
    span = q2 - q1 if np.isfinite(q2 - q1) else 0.0

    def flux_to_alpha(f):
        if not np.isfinite(f) or span < 1e-8:
            return 1.0
        x = (float(f) - q1) / span
        return float(np.clip(0.3 + 0.7 * x, 0.0, 1.0))

    frames_eff = int(min(frames, max(20, t_rel.size)))

    # --- Figure layout (3D top + 2D bottom) ---
    fig = plt.figure(figsize=(7.2, 8.4))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0])

    ax3d = fig.add_subplot(gs[0], projection='3d')
    axlc = fig.add_subplot(gs[1])

    # 3D orbit scaffolding
    th = np.linspace(0, 2*np.pi, 600)
    x_orb = orbit_radius*np.cos(th)
    y_orb = orbit_radius*np.sin(th)
    z_orb = np.zeros_like(th)
    ax3d.plot(x_orb, y_orb, z_orb, lw=1.5)

    # Star (scatter so we can set alpha dynamically)
    star = ax3d.scatter([0],[0],[0], s=600, alpha=1.0)

    # Planet marker (updated every frame)
    planet = ax3d.scatter([orbit_radius],[0],[0], s=60)

    # Nice view box
    lim = orbit_radius*1.25
    ax3d.set_xlim(-lim, lim); ax3d.set_ylim(-lim, lim); ax3d.set_zlim(-lim*0.2, lim*0.2)
    ax3d.set_xlabel("x"); ax3d.set_ylabel("y"); ax3d.set_zlabel("z")
    ax3d.set_title(f"TIC {kic} — schematic 3D orbit", pad=12)
    ax3d.view_init(elev=elev_deg, azim=45)

    # Flux plot
    axlc.plot(t_rel, y, lw=0.6)
    tracker = axlc.axvline(0.0, ls="--", lw=1.0)
    axlc.set_xlim(0, float(period))
    axlc.set_xlabel("Phase within period [days]")
    axlc.set_ylabel("Flux (normalized)")
    axlc.set_title("Measured flux vs phase (PDCSAP)")

    def update(i):
        # Index into folded series
        idx = int((i / max(1, (frames_eff - 1))) * (t_rel.size - 1))
        phase_now = float(t_rel[idx])
        angle = (phase_now / float(period)) * 2*np.pi

        # Planet position on circular orbit (in x–y plane)
        xp = orbit_radius*np.cos(angle)
        yp = orbit_radius*np.sin(angle)
        zp = 0.0

        # Update artists
        planet._offsets3d = ([xp], [yp], [zp])  # updating Path3DCollection
        tracker.set_xdata([phase_now, phase_now])

        # Star brightness follows flux
        a = flux_to_alpha(y[idx])
        star.set_alpha(a)

        # Optional camera rotation for depth perception
        if rotate_camera:
            az = 45 + 360*rot_per_anim*(i/frames_eff)
            ax3d.view_init(elev=elev_deg, azim=az)

        return planet, tracker, star

    anim = FuncAnimation(fig, update, frames=frames_eff, interval=35, blit=False)
    plt.tight_layout()

    if save_path:
        try:
            anim.save(save_path, writer=PillowWriter(fps=25))
            print("Saved animation:", save_path)
        except Exception as e:
            print("Could not save 3D animation. Error:", e)

    if close_after:
        plt.close(fig)

    return anim



In [27]:
# enriched_tess_df must contain: tid (TIC), pl_orbper [days], pl_tranmid [BJD_TDB]
if len(enriched_tess_df) > 0:
    for i, row in enriched_tess_df.iterrows():
        tic = int(row["tid"])
        P   = float(row["pl_orbper"])
        t0_bjd = float(row["pl_tranmid"])     # BJD_TDB from your table

        # TESS light curve time is BTJD = BJD_TDB - 2457000
        t0_btjd = t0_bjd - 2457000.0

        lc = fetch_pdcsap_lightcurve(tic)
        if lc is None:
            print(f"Skipping TIC {tic} (no LC).")
            continue

        print(f"Building 3D animation for TIC {tic} ({i+1}/{len(enriched_tess_df)}) ...")
        gif3d_path = str((out_dir / f"{tic}.gif").resolve())

        anim3d = make_orbit_light_animation_3d(
            tic, lc, P, t0_btjd,   # pass BTJD epoch for TESS data
            frames=300,
            save_path=gif3d_path,
            close_after=True
        )

        print(f"Saved GIF for TIC {tic} → {gif3d_path}")




Building 3D animation for TIC 351601843 (1/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\351601843.gif
Saved GIF for TIC 351601843 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\351601843.gif
Building 3D animation for TIC 394561119 (2/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\394561119.gif
Saved GIF for TIC 394561119 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\394561119.gif
Building 3D animation for TIC 52368076 (3/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\52368076.gif
Saved GIF for TIC 52368076 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\52368076.gif
Building 3D animation for TIC 355867695 (4/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\355867695.gif
Saved GIF for TIC 355867695 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\355867695.gif


Building 3D animation for TIC 417948359 (5/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\417948359.gif
Saved GIF for TIC 417948359 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\417948359.gif
Building 3D animation for TIC 89020549 (6/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\89020549.gif
Saved GIF for TIC 89020549 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\89020549.gif
Building 3D animation for TIC 62483237 (7/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\62483237.gif
Saved GIF for TIC 62483237 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\62483237.gif
Building 3D animation for TIC 428679607 (8/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\428679607.gif
Saved GIF for TIC 428679607 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\428679607.gif
Building 3D animation for TIC 28900646 (9/32) ...
Saved animatio

Building 3D animation for TIC 404505029 (13/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\404505029.gif
Saved GIF for TIC 404505029 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\404505029.gif


Building 3D animation for TIC 27491137 (14/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\27491137.gif
Saved GIF for TIC 27491137 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\27491137.gif
Building 3D animation for TIC 441738827 (15/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\441738827.gif
Saved GIF for TIC 441738827 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\441738827.gif
Building 3D animation for TIC 235678745 (16/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\235678745.gif
Saved GIF for TIC 235678745 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\235678745.gif
Building 3D animation for TIC 392476080 (17/32) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\392476080.gif
Saved GIF for TIC 392476080 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\392476080.gif
Building 3D animation for TIC 88992642 (18/32) ...
Saved

In [19]:
exoid = [
  {"Source": "TESS", "TID/KEPID": 8260536, "TOI/KOI": "5398.01"},
  {"Source": "KOI", "TID/KEPID": 12785320, "TOI/KOI": "K00298.01"},
  {"Source": "TESS", "TID/KEPID": 219016883, "TOI/KOI": "5238.01"},
  {"Source": "KOI", "TID/KEPID": 12644822, "TOI/KOI": "K00791.01"},
  {"Source": "TESS", "TID/KEPID": 100389539, "TOI/KOI": "4791.01"},
  {"Source": "KOI", "TID/KEPID": 12404305, "TOI/KOI": "K00486.01"},
  {"Source": "TESS", "TID/KEPID": 437856897, "TOI/KOI": "4603.01"},
  {"Source": "KOI", "TID/KEPID": 12400538, "TOI/KOI": "K01503.01"},
  {"Source": "TESS", "TID/KEPID": 232608943, "TOI/KOI": "4600.02"},
  {"Source": "KOI", "TID/KEPID": 12206313, "TOI/KOI": "K02714.01"},
  {"Source": "TESS", "TID/KEPID": 354944123, "TOI/KOI": "4342.02"},
  {"Source": "KOI", "TID/KEPID": 12120307, "TOI/KOI": "K02597.01"},
  {"Source": "TESS", "TID/KEPID": 256722647, "TOI/KOI": "4329.01"},
  {"Source": "KOI", "TID/KEPID": 12068975, "TOI/KOI": "K00623.03"},
  {"Source": "TESS", "TID/KEPID": 257060897, "TOI/KOI": "4138.01"},
  {"Source": "KOI", "TID/KEPID": 12061969, "TOI/KOI": "K02061.02"},
  {"Source": "TESS", "TID/KEPID": 289661991, "TOI/KOI": "3807.01"},
  {"Source": "KOI", "TID/KEPID": 12058931, "TOI/KOI": "K00546.02"},
  {"Source": "TESS", "TID/KEPID": 17865622, "TOI/KOI": "3540.01"},
  {"Source": "KOI", "TID/KEPID": 12058204, "TOI/KOI": "K02218.02"},
  {"Source": "TESS", "TID/KEPID": 428699140, "TOI/KOI": "3082.01"},
  {"Source": "KOI", "TID/KEPID": 12058147, "TOI/KOI": "K02072.01"},
  {"Source": "TESS", "TID/KEPID": 361343239, "TOI/KOI": "2977.01"},
  {"Source": "KOI", "TID/KEPID": 11963206, "TOI/KOI": "K02820.01"},
  {"Source": "TESS", "TID/KEPID": 178162579, "TOI/KOI": "2842.01"},
  {"Source": "KOI", "TID/KEPID": 11853878, "TOI/KOI": "K01833.03"},
  {"Source": "TESS", "TID/KEPID": 220076110, "TOI/KOI": "2796.01"},
  {"Source": "KOI", "TID/KEPID": 11807274, "TOI/KOI": "K00262.01"},
  {"Source": "TESS", "TID/KEPID": 258920431, "TOI/KOI": "2567.01"},
  {"Source": "KOI", "TID/KEPID": 11764462, "TOI/KOI": "K01531.01"},
  {"Source": "KOI", "TID/KEPID": 11754553, "TOI/KOI": "K00775.03"},
  {"Source": "KOI", "TID/KEPID": 11720424, "TOI/KOI": "K03116.01"},
  {"Source": "KOI", "TID/KEPID": 11718144, "TOI/KOI": "K02310.01"},
  {"Source": "KOI", "TID/KEPID": 11656918, "TOI/KOI": "K01945.01"},
  {"Source": "KOI", "TID/KEPID": 11656246, "TOI/KOI": "K01532.01"}
]

# turn into a dataframe
exo_df = pd.DataFrame(exoid)

# bring in the koi data
exo_koi_data_path = "data/Kepler Object of Interest.csv"
exo_koi_df = pd.read_csv(exo_koi_data_path, usecols=["kepid", "kepoi_name", "koi_period", "koi_time0bk"])

# bring in the tess data
exo_tess_data_path = "data/TESS Project Candidates.csv"
exo_tess_df = pd.read_csv(exo_tess_data_path, usecols=["tid", "toi", "pl_orbper", "pl_tranmid"])

# rename both exo dfs to union on koi/toi
exo_koi_df = exo_koi_df.rename(columns={
    "kepid": "ID",
    "kepoi_name": "EXOName",
    "koi_period": "Period",
    "koi_time0bk": "Epoch"
})  

exo_tess_df = exo_tess_df.rename(columns={
    "tid": "ID",  
    "toi": "EXOName",
    "pl_orbper": "Period",
    "pl_tranmid": "Epoch"
})  

# turn EXOName in exo_tess_df to string type
exo_tess_df["EXOName"] = exo_tess_df["EXOName"].astype(str)

# union both dataframes
enriched_exo_df = pd.concat([exo_koi_df, exo_tess_df], ignore_index=True) 

# join with exoid on ID and EXOName
enriched_exo_df = pd.merge(exo_df, enriched_exo_df, left_on=["TID/KEPID", "TOI/KOI"], right_on=["ID", "EXOName"], how="left")

# slice the columns to TID, EXOName, Period, and Epoch
enriched_exo_df = enriched_exo_df[["Source", "TID/KEPID", "EXOName", "Period", "Epoch"]]

enriched_exo_df

,Source,TID/KEPID,EXOName,Period,Epoch
0,TESS,8260536,5398.01,10.590923,2.459616e+06
1,KOI,12785320,K00298.01,19.963672,1.783081e+02
2,TESS,219016883,5238.01,4.872190,2.459662e+06
3,KOI,12644822,K00791.01,12.611904,1.808923e+02
4,TESS,100389539,4791.01,4.280977,2.459246e+06
5,KOI,12404305,K00486.01,22.183323,1.694962e+02
6,TESS,437856897,4603.01,7.245600,2.460274e+06
7,KOI,12400538,K01503.01,150.241258,1.382901e+02
8,TESS,232608943,4600.02,482.819100,2.459752e+06
9,KOI,12206313,K02714.01,14.383227,1.405765e+02


In [16]:
# view enriched df info
enriched_exo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TID/KEPID  35 non-null     int64  
 1   EXOName    35 non-null     object 
 2   Period     35 non-null     float64
 3   Epoch      35 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1.2+ KB


In [17]:
# view 
exo_koi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       9564 non-null   int64  
 1   EXOName  9564 non-null   object 
 2   Period   9564 non-null   float64
 3   Epoch    9564 non-null   float64
dtypes: float64(2), int64(1), object(1)
memory usage: 299.0+ KB


In [18]:
# view 
exo_tess_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7703 entries, 0 to 7702
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   EXOName  7703 non-null   object 
 1   ID       7703 non-null   int64  
 2   Epoch    7703 non-null   float64
 3   Period   7596 non-null   float64
dtypes: float64(2), int64(1), object(1)
memory usage: 240.8+ KB


In [ ]:
# --- deps (unchanged) ---
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation, PillowWriter
import lightkurve as lk

# Prepare both output roots (we'll choose per-row in your loop)
Path("gifs/tess").mkdir(parents=True, exist_ok=True)
Path("gifs/koi").mkdir(parents=True, exist_ok=True)

def fetch_pdcsap_lightcurve(tic_id: int):
    """Download & return a stitched, normalized, flattened TESS light curve for a TIC ID.
    Tries SPOC/QLP across common cadences; falls back to any author; then TESSCut FFIs.
    """
    try:
        # Try common authors and cadences first
        for author in ("SPOC", "QLP"):
            # None means "any cadence"; otherwise try specific exptimes too
            for exptime in (20, 120, 600, 1800, None):
                sr = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS",
                                          author=author, exptime=exptime)
                if len(sr) == 0:
                    continue
                lc_col = sr.download_all()
                if lc_col is None or len(lc_col) == 0:
                    continue
                lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
                return lc

        # Fallback: any author/cadence
        sr_any = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")
        if len(sr_any) > 0:
            lc_col = sr_any.download_all()
            if lc_col is not None and len(lc_col) > 0:
                lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
                return lc

        # Last resort: TESSCut (FFI cutouts → light curve)
        sr_cut = lk.search_tesscut(f"TIC {tic_id}")
        if len(sr_cut) > 0:
            # Download first available sector cutout and extract LC
            tpf = sr_cut[0].download(cutout_size=15)
            if tpf is not None:
                lc = (tpf.to_lightcurve(aperture_mask="pipeline")
                          .remove_nans().normalize().flatten(window_length=401))
                return lc

        print(f"No usable TESS light curve found for TIC {tic_id} (SPOC, QLP, any, TESSCut).")
        return None
    except Exception as e:
        print(f"Error fetching LC for TIC {tic_id}: {e}")
        return None


def fetch_kepler_lightcurve(kic_id: int):
    """Download & return a stitched, normalized, flattened Kepler light curve for a KIC ID."""
    try:
        # Try canonical Kepler cadences; fall back to "any"
        for exptime in (60, 1800, None):  # short, long, any
            sr = lk.search_lightcurve(f"KIC {kic_id}", mission="Kepler", exptime=exptime)
            if len(sr) == 0:
                continue
            lc_col = sr.download_all()
            if lc_col is None or len(lc_col) == 0:
                continue
            lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
            return lc

        # As a final fallback, try without exptime constraint
        sr_any = lk.search_lightcurve(f"KIC {kic_id}", mission="Kepler")
        if len(sr_any) > 0:
            lc_col = sr_any.download_all()
            if lc_col is not None and len(lc_col) > 0:
                lc = lc_col.stitch().remove_nans().normalize().flatten(window_length=401)
                return lc

        print(f"No usable Kepler light curve found for KIC {kic_id}.")
        return None
    except Exception as e:
        print(f"Error fetching LC for KIC {kic_id}: {e}")
        return None


def make_orbit_light_animation_3d(
    target_id: int, lc, period: float, t0: float,
    frames: int = 300, orbit_radius: float = 3.0, save_path=None,
    rotate_camera=True, rot_per_anim=0.6, elev_deg=22,
    close_after=False, id_label: str = "TIC"
):
    """
    3D orbit (top) + measured PDCSAP/SAP flux vs phase (bottom), synchronized.
    Pass an epoch (t0) already expressed in the mission's time base:
      - TESS: BTJD (BJD_TDB - 2457000)
      - Kepler: BKJD (BJD_TDB - 2454833)
    """
    if lc is None:
        print("No light curve available for animation.")
        return None

    # --- Prepare arrays (plain floats, handle masked/quantities) ---
    t = np.asarray(getattr(lc.time, "value", lc.time), dtype=float)
    y = getattr(lc, "flux", None)
    if y is None:
        print("No flux in light curve.")
        return None
    y = getattr(y, "value", y)
    y = np.ma.getdata(y).astype(float, copy=False)

    # drop non-finite pairs up-front
    m = np.isfinite(t) & np.isfinite(y)
    t, y = t[m], y[m]

    if not np.isfinite(period) or period <= 0 or t.size < 10 or y.size < 10:
        print("Invalid LC/period for animation.")
        return None

    # normalize flux to ~1.0 median (helps alpha mapping)
    med = np.nanmedian(y)
    if np.isfinite(med) and med != 0:
        y = y / med

    # Phase-fold & sort for smooth traversal (epoch already in correct time base)
    t_rel = (t - float(t0)) % float(period)
    order = np.argsort(t_rel)
    t_rel, y = t_rel[order], y[order]
    if t_rel.size < 10:
        print("Too few samples to animate.")
        return None

    # Robust brightness -> alpha mapping
    finite_y = y[np.isfinite(y)]
    if finite_y.size < 10:
        print("Too few finite flux samples to animate.")
        return None
    q1, q2 = np.nanpercentile(finite_y, [1, 99])
    span = q2 - q1 if np.isfinite(q2 - q1) else 0.0

    def flux_to_alpha(f):
        if not np.isfinite(f) or span < 1e-8:
            return 1.0
        x = (float(f) - q1) / span
        return float(np.clip(0.3 + 0.7 * x, 0.0, 1.0))

    frames_eff = int(min(frames, max(20, t_rel.size)))

    # --- Figure layout (3D top + 2D bottom) ---
    fig = plt.figure(figsize=(7.2, 8.4))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0])

    ax3d = fig.add_subplot(gs[0], projection='3d')
    axlc = fig.add_subplot(gs[1])

    # 3D orbit scaffolding
    th = np.linspace(0, 2*np.pi, 600)
    x_orb = orbit_radius*np.cos(th)
    y_orb = orbit_radius*np.sin(th)
    z_orb = np.zeros_like(th)
    ax3d.plot(x_orb, y_orb, z_orb, lw=1.5)

    # Star (scatter so we can set alpha dynamically)
    star = ax3d.scatter([0],[0],[0], s=600, alpha=1.0)

    # Planet marker (updated every frame)
    planet = ax3d.scatter([orbit_radius],[0],[0], s=60)

    # Nice view box
    lim = orbit_radius*1.25
    ax3d.set_xlim(-lim, lim); ax3d.set_ylim(-lim, lim); ax3d.set_zlim(-lim*0.2, lim*0.2)
    ax3d.set_xlabel("x"); ax3d.set_ylabel("y"); ax3d.set_zlabel("z")
    ax3d.set_title(f"{id_label} {target_id} — schematic 3D orbit", pad=12)
    ax3d.view_init(elev=elev_deg, azim=45)

    # Flux plot
    axlc.plot(t_rel, y, lw=0.6)
    tracker = axlc.axvline(0.0, ls="--", lw=1.0)
    axlc.set_xlim(0, float(period))
    axlc.set_xlabel("Phase within period [days]")
    axlc.set_ylabel("Flux (normalized)")
    axlc.set_title("Measured flux vs phase")

    def update(i):
        # Index into folded series
        idx = int((i / max(1, (frames_eff - 1))) * (t_rel.size - 1))
        phase_now = float(t_rel[idx])
        angle = (phase_now / float(period)) * 2*np.pi

        # Planet position on circular orbit (in x–y plane)
        xp = orbit_radius*np.cos(angle)
        yp = orbit_radius*np.sin(angle)
        zp = 0.0

        # Update artists
        planet._offsets3d = ([xp], [yp], [zp])  # updating Path3DCollection
        tracker.set_xdata([phase_now, phase_now])

        # Star brightness follows flux
        a = flux_to_alpha(y[idx])
        star.set_alpha(a)

        # Optional camera rotation for depth perception
        if rotate_camera:
            az = 45 + 360*rot_per_anim*(i/frames_eff)
            ax3d.view_init(elev=elev_deg, azim=az)

        return planet, tracker, star

    anim = FuncAnimation(fig, update, frames=frames_eff, interval=35, blit=False)
    plt.tight_layout()

    if save_path:
        try:
            anim.save(save_path, writer=PillowWriter(fps=25))
            print("Saved animation:", save_path)
        except Exception as e:
            print("Could not save 3D animation. Error:", e)

    if close_after:
        plt.close(fig)

    return anim


In [23]:
# enriched_exo_df must contain: Source, TID/KEPID, Period [days], Epoch
# (Epoch time base: TESS= BJD_TDB → convert to BTJD; KOI/Kepler = BKJD already)
from pathlib import Path
import pandas as pd

tess_dir = Path("gifs/tess"); tess_dir.mkdir(parents=True, exist_ok=True)
koi_dir  = Path("gifs/koi");  koi_dir.mkdir(parents=True, exist_ok=True)

if len(enriched_exo_df) > 0:
    for i, row in enriched_exo_df.iterrows():
        src = str(row["Source"]).strip().upper()

        # basic parsing/sanity
        try:
            obj_id = int(row["TID/KEPID"])
            P      = float(row["Period"])
            t0     = float(row["Epoch"])
        except Exception:
            print(f"Skipping row {i}: non-numeric ID/Period/Epoch.")
            continue
        if P <= 0:
            print(f"Skipping row {i}: invalid period {P}.")
            continue

        if src == "TESS":
            # TESS: convert BJD_TDB epoch → BTJD
            t0_pass = t0 - 2457000.0
            lc = fetch_pdcsap_lightcurve(obj_id)
            out_dir = tess_dir
            id_label = "TIC"
            fname = f"{obj_id}.gif"
        elif src == "KOI":
            # Kepler/KOI: koi_time0bk is already BKJD
            t0_pass = t0
            lc = fetch_kepler_lightcurve(obj_id)
            out_dir = koi_dir
            id_label = "KIC"
            fname = f"{obj_id}.gif"
        else:
            print(f"Skipping row {i}: unknown Source '{row['Source']}'.")
            continue

        if lc is None:
            print(f"Skipping {id_label} {obj_id} (no LC).")
            continue

        print(f"Building 3D animation for {id_label} {obj_id} ({i+1}/{len(enriched_exo_df)}) ...")
        gif3d_path = str((out_dir / fname).resolve())

        anim3d = make_orbit_light_animation_3d(
            obj_id, lc, P, t0_pass,             # pass mission-appropriate epoch
            frames=300,
            save_path=gif3d_path,
            close_after=True,
            id_label=id_label
        )

        if anim3d is not None:
            print(f"Saved GIF for {id_label} {obj_id} → {gif3d_path}")


Building 3D animation for TIC 8260536 (1/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\8260536.gif
Saved GIF for TIC 8260536 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\8260536.gif
Building 3D animation for KIC 12785320 (2/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12785320.gif
Saved GIF for KIC 12785320 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12785320.gif
Building 3D animation for TIC 219016883 (3/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\219016883.gif
Saved GIF for TIC 219016883 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\219016883.gif
Building 3D animation for KIC 12644822 (4/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12644822.gif
Saved GIF for KIC 12644822 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\12644822.gif
Building 3D animation for TIC 100389539 (5/35) ...
Saved animation: C:\Roger

No data found for target "TIC 220076110".
No data found for target "TIC 220076110".
No data found for target "TIC 220076110".
No data found for target "TIC 220076110".
No data found for target "TIC 220076110".


Building 3D animation for TIC 220076110 (27/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\220076110.gif
Saved GIF for TIC 220076110 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\220076110.gif
Building 3D animation for KIC 11807274 (28/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11807274.gif
Saved GIF for KIC 11807274 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11807274.gif
Building 3D animation for TIC 258920431 (29/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\258920431.gif
Saved GIF for TIC 258920431 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\tess\258920431.gif
Building 3D animation for KIC 11764462 (30/35) ...
Saved animation: C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11764462.gif
Saved GIF for KIC 11764462 → C:\Roger\goals\creating\blog\NASA-Space-Apps\gifs\koi\11764462.gif
Building 3D animation for KIC 11754553 (31/35) ...
Saved animati